**Notes from Bety:**

- Be sure to experiment with large populations and niterations ($\mathcal{O}(1000)$) early, as equations from smaller experiments don't always reflect larger runs with the same hyperparameters/settings (new "regimes" sometimes emerge). This includes new variables appearing as important (in place of others earlier in the run) or complicated expressions with many variables replaced by more nonlinear functions of fewer variables.
- `exp()` or more nonlinear operators (higher powers) require more constraints, and often nested constraints (i.e. `'exp':{'exp':0}`). In some cases, multiple nested (i.e. `'exp':{'exp':{'exp':0} ...}`) functions like $x_1^{x_2}$ and $x_1*exp(x_2^{x_3}$) can be dealt with by adjusting the `nested_constraints` (note this is more precise than using the `complexity_of_operators` parameter, which tends to get rid of certain operators altogether). Adjusting `complexity_of_variables` parameter can also help address these issues (especially when the power operator relies on multiple variables). 
- Arguments in the power operator tend to become complicated, the constraint `constraints={'^': (-1,1)}` helps limit the complexity of the power argument (you can experiment with more complex args `constraints={'^':(-1,2)}`). 
- The ratio `complexity_of_variables`/`complexity_of_constants` also is important (not just each individually) . Using `complexity_of_variables = 1` (or too small) leads to complicated combinations of variables early on often with the power operator, such as $x_1^{(x_2*x_3)}$.
- Note the best `complexity_of_variables` value will likely depend on number of variables (chosen for 8 input variables in this example).
- The native PySR `select_k_features` parameter didn't work well for me. For reducing number of variables in the final equations, increasing the complexity of variables and doing very large runs can help identify which variables are important.
- Batching is critical to make datasets of size $> ~\mathcal{O}(10^4)$ manageable. Compute is generally better spent on longer runs (with smaller batches) versus shorter runs with no batching (and more data points). 
- Refer to https://astroautomata.com/PySR/tuning/, although note some functionality mentioned is outdated or no longer supported
- Experiment with different variable scalings (e.g., z-scoring a strictly positive variable $x_1$ might get rid of any $x_1^a$ where $a<1$). In theory and depending on the complexity settings, the algorithm should recover $(x_1 + \mathrm{offset})^a$ but in practice it doesn't always and so scaling choices can change the form and performance of the equations.
- See https://ai.damtp.cam.ac.uk/pysr/api/ for the many options in PySRRegressor

In [ ]:
import os
os.environ['JULIA_DEPOT_PATH'] = '/pscratch/sd/s/sferrett/.julia'
os.environ['PYTHON_JULIAPKG_PROJECT'] = '/pscratch/sd/s/sferrett/.julia/environments/pyjuliapkg'
import gc
import json
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import xarray as xr
import pandas as pd
import proplot as pplt
from pysr import PySRRegressor, jl
jl.seval('safe_pow(x, y) = abs(x)^y')
pplt.rc.update({'figure.dpi':100})

In [ ]:
# ════════════════════════════════════════════════════════════════════
# Cell 1: Configuration
# ════════════════════════════════════════════════════════════════════
with open('../scripts/configs.json', 'r', encoding='utf-8') as f:
    CONFIGS = json.load(f)

SPLITSDIR = CONFIGS['filepaths']['splits']
PREDSDIR  = CONFIGS['filepaths']['predictions']
FEATSDIR  = CONFIGS['filepaths']['features']
MODELSDIR = CONFIGS['filepaths']['models']
TARGETVAR = CONFIGS['variables']['target']

# ── Select an SR run from the config ──
SR_RUNS = CONFIGS['experiments']['sr']['runs']
print(f"Available SR runs: {list(SR_RUNS.keys())}")

RUNNAME   = list(SR_RUNS.keys())[0]  # <-- change index or hardcode name
RUNCONFIG = SR_RUNS[RUNNAME]

USE_FIELDVARS  = RUNCONFIG['fieldvars']
USE_LOCALVARS  = RUNCONFIG['localvars']
FEATURES_FROM  = RUNCONFIG['features_from']  # NN run whose kernel features we use
ALL_PREDICTOR_NAMES = USE_FIELDVARS + USE_LOCALVARS

print(f"\nRun: {RUNNAME}")
print(f"  Field vars (kernel-integrated from '{FEATURES_FROM}'): {USE_FIELDVARS}")
print(f"  Local vars (from normalized splits): {USE_LOCALVARS}")
print(f"  Target: {TARGETVAR}")
print(f"  All predictors: {ALL_PREDICTOR_NAMES}")

In [ ]:
# ════════════════════════════════════════════════════════════════════
# Cell 2: Load Data
# ════════════════════════════════════════════════════════════════════
def load_split(splitname, fieldvars, localvars, targetvar,
               splitsdir, featsdir, features_from, seed_idx=0):
    """
    Load a data split for symbolic regression.

    Field variables come from kernel-integrated features produced by a
    trained NN model (NetCDF in FEATSDIR).  Local variables and the target
    come from the normalized splits (HDF5/h5netcdf in SPLITSDIR).

    Returns
    -------
    X : pd.DataFrame  — predictors (fieldvars + localvars), one row per sample
    y : np.ndarray     — target values, shape (nsamp,)
    refda : xr.DataArray — reference (time, lat, lon) grid for reconstruction
    """
    # ── Kernel-integrated features from the NN model ──
    feat_path = os.path.join(featsdir, f'{features_from}_{splitname}_features.nc')
    feat_ds   = xr.open_dataset(feat_path, engine='h5netcdf')

    # ── Normalized split for local vars + target ──
    split_path = os.path.join(splitsdir, f'norm_{splitname}.h5')
    split_ds   = xr.open_dataset(split_path, engine='h5netcdf')

    refda = split_ds[targetvar].transpose('time', 'lat', 'lon')
    ntime = split_ds.sizes['time']

    # ── Build predictor columns ──
    columns = {}

    for var in fieldvars:
        # Features dims: (time, lat, lon, seed) — select one seed, flatten
        da = feat_ds[var].isel(seed=seed_idx).transpose('time', 'lat', 'lon')
        columns[var] = da.values.ravel()

    for var in localvars:
        da = split_ds[var]
        if 'time' in da.dims:
            arr = da.transpose('time', 'lat', 'lon').values
        else:
            # Static field (e.g. land fraction) — broadcast across time
            arr = np.tile(da.values, (ntime, 1, 1))
        columns[var] = arr.ravel()

    X = pd.DataFrame(columns)
    y = refda.values.ravel()

    feat_ds.close()
    split_ds.close()

    return X, y, refda


# ── Load train + valid splits ──
X_train, y_train, refda_train = load_split(
    'train', USE_FIELDVARS, USE_LOCALVARS, TARGETVAR,
    SPLITSDIR, FEATSDIR, FEATURES_FROM)

X_valid, y_valid, refda_valid = load_split(
    'valid', USE_FIELDVARS, USE_LOCALVARS, TARGETVAR,
    SPLITSDIR, FEATSDIR, FEATURES_FROM)

print(f"Train: X={X_train.shape}, y={y_train.shape}")
print(f"Valid: X={X_valid.shape}, y={y_valid.shape}")

In [ ]:
# ════════════════════════════════════════════════════════════════════
# Cell 4: Clean Data
# ════════════════════════════════════════════════════════════════════
def clean_Xy(X, y):
    """Remove NaN and Inf from paired X, y data."""
    df = X.copy()
    df['__target__'] = y
    df = df.replace([np.inf, -np.inf], np.nan).dropna()
    y_clean = df.pop('__target__').values
    X_clean = df
    return X_clean, y_clean

X_train, y_train = clean_Xy(X_train, y_train)
X_valid, y_valid = clean_Xy(X_valid, y_valid)

print(f"Train after cleaning: X={X_train.shape}, y={y_train.shape}")
print(f"Valid after cleaning: X={X_valid.shape}, y={y_valid.shape}")
print(f"\nTrain X summary:\n{X_train.describe()}")
print(f"\nTrain y summary: mean={y_train.mean():.6f}, std={y_train.std():.6f}, "
      f"min={y_train.min():.6f}, max={y_train.max():.6f}")

In [ ]:
# ════════════════════════════════════════════════════════════════════
# Cell 5: Subsample for PySR
# ════════════════════════════════════════════════════════════════════
# PySR does NOT benefit from massive datasets — it's symbolic search,
# not gradient descent. 5,000–10,000 well-chosen points is plenty.
# Your collaborator used 5,000.

SUBSET_SIZE = 5000
SEED = 42

np.random.seed(SEED)

if len(y_train) > SUBSET_SIZE:
    subset_idx = np.random.choice(len(y_train), size=SUBSET_SIZE, replace=False)
    X_sub = X_train.iloc[subset_idx].reset_index(drop=True)
    y_sub = y_train[subset_idx]
    print(f"Subsampled {SUBSET_SIZE} from {len(y_train)} training points")
else:
    X_sub = X_train.reset_index(drop=True)
    y_sub = y_train
    print(f"Using all {len(y_train)} training points (< {SUBSET_SIZE})")

print(f"X_sub: {X_sub.shape}, y_sub: {y_sub.shape}")

In [ ]:
# ════════════════════════════════════════════════════════════════════
# Cell 5: Quick Data Visualization
# ════════════════════════════════════════════════════════════════════
n_vars = len(ALL_PREDICTOR_NAMES)
fig, axes = pplt.subplots(ncols=n_vars, refwidth=3.5, refheight=3)

for i, var in enumerate(ALL_PREDICTOR_NAMES):
    ax = axes[i]
    ax.scatter(X_sub[var], y_sub, s=1, alpha=0.3)
    ax.format(xlabel=var, ylabel=TARGETVAR if i == 0 else '')

fig.suptitle(f'{TARGETVAR} vs Predictors (n={len(y_sub)})')
pplt.show()

In [ ]:
# ════════════════════════════════════════════════════════════════════
# Cell 7: PySR Model Configuration
# ════════════════════════════════════════════════════════════════════

# ── Operator complexity tiers ──
# Tier 1 (cheap):   +, -, *
# Tier 2 (medium):  /
# Tier 3 (costly):  safe_pow, exp, log

opcomplexity = {
    '+': 1, '-': 1, '*': 1,
    '/': 3,
    'safe_pow': 3,
    'exp': 4, 'log': 4,
}

# ── Search mode ──
# "test"  : quick sanity check (~1 min)
# "medium": decent exploration (~30 min)
# "full"  : production run (~hours)

SEARCH_MODE = "test"

search_params = {
    "test": dict(
        niterations=5,
        populations=5,
        population_size=33,
        maxsize=15,
        timeout_in_seconds=300,
    ),
    "medium": dict(
        niterations=100,
        populations=20,
        population_size=50,
        maxsize=30,
        timeout_in_seconds=1800,
    ),
    "full": dict(
        niterations=10_000_000,  # Effectively infinite — use timeout
        populations=30,
        population_size=100,
        maxsize=50,
        timeout_in_seconds=int(60 * 60 * 7.5),  # 7.5 hours
    ),
}

params = search_params[SEARCH_MODE]
print(f"Search mode: {SEARCH_MODE}")
print(f"Parameters: {params}")

In [ ]:
# ════════════════════════════════════════════════════════════════════
# Cell 7: Build and Run PySR Model
# ════════════════════════════════════════════════════════════════════
import time

OUTDIR = os.path.join(MODELSDIR, 'sr', RUNNAME)
os.makedirs(OUTDIR, exist_ok=True)

model = PySRRegressor(
    # Search size
    niterations=params['niterations'],
    populations=params['populations'],
    population_size=params['population_size'],

    # Operators
    binary_operators=['+', '-', '*', '/', 'safe_pow'],
    unary_operators=['exp', 'log'],

    # Complexity control
    complexity_of_operators=opcomplexity,
    complexity_of_variables=2,
    complexity_of_constants=1,
    maxsize=params['maxsize'],
    maxdepth=4,  # Prevent deep nesting (from collaborator)

    # Constraints
    constraints={'safe_pow': (-1, 1)},
    nested_constraints={
        'exp': {'exp': 0, 'log': 0, 'safe_pow': 0},
        'safe_pow': {'safe_pow': 0},
        'log': {'log': 0, 'exp': 0},
    },

    # Sympy mapping for safe_pow
    extra_sympy_mappings={'safe_pow': lambda x, y: x**y},

    # Loss
    loss='loss(x, y) = (x - y)^2',
    model_selection='best',

    # Data handling
    batching=True if len(y_sub) > 2000 else False,
    batch_size=min(2000, len(y_sub)),

    # Compute
    random_state=SEED,
    deterministic=True,
    multithreading=False,      # Set True for production
    procs=0,

    # Output
    tempdir=OUTDIR,
    temp_equation_file=True,
    delete_tempfiles=False,

    # Safety
    timeout_in_seconds=params['timeout_in_seconds'],
    progress=True,
)

t0 = time.time()
model.fit(X_sub.values, y_sub, variable_names=ALL_PREDICTOR_NAMES)
elapsed = time.time() - t0

print(f"\nSearch completed in {elapsed/60:.1f} minutes")
print(model)

In [ ]:
# ════════════════════════════════════════════════════════════════════
# Cell 9: Results — Equation Table
# ════════════════════════════════════════════════════════════════════
equations = model.equations_
print(equations[['equation', 'complexity', 'loss', 'score']].to_string())

In [ ]:
# ════════════════════════════════════════════════════════════════════
# Cell 9b: Evaluate Best Equation
# ════════════════════════════════════════════════════════════════════
from sklearn.metrics import mean_squared_error

best_eq = model.get_best()

y_pred_train = model.predict(X_train.values)
y_pred_valid = model.predict(X_valid.values)

mse_train = mean_squared_error(y_train, y_pred_train)
mse_valid = mean_squared_error(y_valid, y_pred_valid)

print(f"Best equation: {best_eq['equation']}")
print(f"  Complexity: {best_eq['complexity']}")
print(f"  MSE train:  {mse_train:.6f}")
print(f"  MSE valid:  {mse_valid:.6f}")

In [ ]:
# ════════════════════════════════════════════════════════════════════
# Cell 9: Results — Pareto Frontier
# ════════════════════════════════════════════════════════════════════
fig, ax = pplt.subplots(refwidth=5, refheight=3.5)
ax.scatter(equations['complexity'], equations['loss'], zorder=5)

ax.format(
    xlim=(0, equations['complexity'].max() + 3),
    ylim=(equations['loss'].min() * 0.9, equations['loss'].max() * 1.1),
    xlabel='Complexity', ylabel='MSE Loss',
    title='Pareto Frontier of Discovered Equations')

for i, row in equations.iterrows():
    label = str(row['equation'])[:50]
    ax.annotate(label,
                xy=(row['complexity'], row['loss']),
                xytext=(5, 5), textcoords='offset points',
                fontsize=5, clip_on=True)

fig.savefig(os.path.join(OUTDIR, 'pareto_frontier.png'), dpi=150)
pplt.show()

In [ ]:
# ════════════════════════════════════════════════════════════════════
# Cell 11: Save Results
# ════════════════════════════════════════════════════════════════════
import pickle
from scripts.data.classes import PredictionWriter

# ── Save PySR model (pickle) ──
model_path = os.path.join(OUTDIR, f'{RUNNAME}.pkl')
with open(model_path, 'wb') as f:
    pickle.dump(model, f)

# ── Save equations table ──
equations.to_csv(os.path.join(OUTDIR, 'equations.csv'), index=False)

# ── Save grid-mapped predictions (same format as NN/POD) ──
writer = PredictionWriter(SPLITSDIR, targetvar=TARGETVAR)

for splitname, y_pred, refda in [('train', y_pred_train, refda_train),
                                  ('valid', y_pred_valid, refda_valid)]:
    # Recompute the valid mask to map flat predictions back onto the grid
    X_raw, y_raw, _ = load_split(
        splitname, USE_FIELDVARS, USE_LOCALVARS, TARGETVAR,
        SPLITSDIR, FEATSDIR, FEATURES_FROM)
    valid = np.isfinite(X_raw).all(axis=1).values & np.isfinite(y_raw)
    pred_arr = writer.predictions_to_array(y_pred, valid, refda)
    pred_ds  = writer.predictions_to_dataset(
        pred_arr[..., np.newaxis], refda)  # add seed dim
    writer.save(RUNNAME, pred_ds, 'predictions', splitname, PREDSDIR)

# ── Save run metadata ──
metadata = {
    'run': RUNNAME,
    'features_from': FEATURES_FROM,
    'predictors': ALL_PREDICTOR_NAMES,
    'target': TARGETVAR,
    'subset_size': len(y_sub),
    'train_size': len(y_train),
    'valid_size': len(y_valid),
    'search_mode': SEARCH_MODE,
    'search_params': params,
    'mse_train': float(mse_train),
    'mse_valid': float(mse_valid),
    'best_equation': str(best_eq['equation']),
    'elapsed_minutes': elapsed / 60,
    'seed': SEED,
}
with open(os.path.join(OUTDIR, 'metadata.json'), 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"Model saved to: {model_path}")
print(f"Predictions saved to: {PREDSDIR}")
print(f"Best equation: {best_eq['equation']}")

gc.collect()

In [2]:
# FILEDIR = '/global/cfs/cdirs/m4334/sferrett/monsoon-discovery/data/interim'
# x  = xr.open_dataarray(f'{FILEDIR}/cape.nc').isel(time=slice(0,5),lat=slice(0,5),lon=slice(0,5)).values.ravel()
# y  = xr.open_dataarray(f'{FILEDIR}/bl.nc').isel(time=slice(0,5),lat=slice(0,5),lon=slice(0,5)).values.ravel()
# df = pd.DataFrame({'x':x,'y':y})

# df = df.replace([np.inf,-np.inf],np.nan).dropna()
# print(f'Data shape after cleaning: {df.shape}')
# print(df.describe())

# fig,ax = pplt.subplots(nrows=1,ncols=1)
# ax.format(xlabel='$CAPE_L$ (K)',ylabel='$B_L$ (m/s$^2$)')
# ax.scatter(df['x'],df['y'],alpha=0.6)
# pplt.show()

# model = PySRRegressor(
#     niterations=3,
#     populations=3,
#     population_size=33,
#     tournament_selection_n=2,
#     binary_operators=['+','-','*','/','safe_pow'],
#     unary_operators=['exp','log'],
#     complexity_of_operators={'+':1,'*':1,'-':1,'/':3,'safe_pow':3,'exp':4,'log':4},
#     complexity_of_variables=2,
#     complexity_of_constants=1,
#     maxsize=10,
#     constraints={'safe_pow':(-1,1)},
#     nested_constraints={
#         'exp':{'exp':0,'log':0,'safe_pow':0},
#         'safe_pow':{'safe_pow':0},
#         'log':{'log': 0,'exp':0}},  
#     extra_sympy_mappings={'safe_pow':lambda x,y:x**y},
#     batching=False,
#     batch_size=100,
#     loss='loss(x, y) = (x - y)^2',
#     model_selection='best',
#     random_state=42,
#     deterministic=True,
#     multithreading=False,
#     procs=0,
#     timeout_in_seconds=300)

# model.fit(df[['x']].values, df['y'].values)
# print(model)

# equations = model.equations_
# print(equations[['equation','complexity','loss','score']].head(10))

# best  = model.get_best()
# eqstr = best['sympy_format']
# ypred = best['lambda_format'](df[['x']].values)

# fig,ax = pplt.subplots(nrows=1,ncols=1,refheight=2,refwidth=2)
# ax.format(title='Pareto Frontier of Discovered Equations',xlabel='Complexity',ylabel='MSE Loss')
# ax.scatter(equations['complexity'],equations['loss'])
# for i,row in equations.iterrows():
#     ax.annotate(row['equation'],xy=(row['complexity'],row['loss']),xytext=(5,5),textcoords='offset points',fontsize=6)
# pplt.show()

# fig,ax = pplt.subplots(nrows=1,ncols=1,refwidth=2.5)
# ax.format(title=f'Best PySR Equation: {eqstr}',grid=True,xlabel='$y_{true}$',ylabel='$y_{pred}$')
# ax.plot(df['y'],df['y'],color='k',linestyle='--')
# ax.scatter(df['y'],ypred,alpha=0.6)
# pplt.show()